# Visualization of shape features

Shape features of a subset of dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

from heavyedge import ProfileData

X = pd.read_csv("../../benchmarks/v1/dimless.csv")
y = pd.read_csv("../../benchmarks/v1/shape_features/minirocket.sigmoid.csv")
idxs = np.load("../../benchmarks/v1/index.npy")
data = pd.concat([X, y], axis=1).iloc[idxs]

## Visual example of phi

In [ ]:
target_idxs = np.load("../../benchmarks/v1/phi-index.npy")

with ProfileData("../../benchmarks/v1/phi-profiles.h5") as profile_data:
    x = profile_data.x()
    Ys, Ls, _ = profile_data[:]

phi = y["phi"][target_idxs]
N = len(target_idxs)
colors = [plt.cm.tab10(i) for i in range(N)]

sort_idx = np.argsort(phi)
target_idxs = target_idxs[sort_idx]
phi = phi.iloc[sort_idx]
Ys = [Ys[i] for i in sort_idx]
Ls = [Ls[i] for i in sort_idx]

In [ ]:
fig = plt.figure(figsize=(6, 4))
fig.set_layout_engine("none")
gs = fig.add_gridspec(
    2,
    N,
    hspace=0.35,
    height_ratios=[1, 2],
    left=0.12,
    right=0.97,
    bottom=0.12,
    top=0.88,
)

ax_profiles = [fig.add_subplot(gs[0, i]) for i in range(N)]
fig.supxlabel("Edge shapes", y=0.99, va="top")
for i, idx in enumerate(target_idxs):
    Y = Ys[i]
    L = Ls[i]
    Y_norm = Y[:L] / Y[:L].max()
    ax_profiles[i].plot(x[:L], Y_norm, lw=0.8, color=colors[i])
    ax_profiles[i].set_xticks([])
    ax_profiles[i].set_yticks([])
    ax_profiles[i].axis("off")

ax_bot = fig.add_subplot(gs[1, :])
ax_bot.hist(y["phi"], bins=40, color="gray", alpha=0.7, edgecolor="white")
for i, idx in enumerate(target_idxs):
    ax_bot.axvline(phi[idx], ls="--", color=colors[i])
ax_bot.set_xlabel(r"$\phi_\text{class}$")
ax_bot.spines[["top", "right"]].set_visible(False)
ax_bot.tick_params(axis="both", width=0.3)
fig.show()

## Shape features

In [ ]:
slurries = ["G50", "G45", "G40", "G40+IPA"]

Cas = data["capillary_number"].unique()
Cas_sorted = np.sort(Cas)
cmap = plt.get_cmap("viridis", len(Cas_sorted))
norm = mcolors.BoundaryNorm(
    np.concatenate(
        [
            [Cas_sorted[0] * 0.9],
            (Cas_sorted[:-1] + Cas_sorted[1:]) / 2,
            [Cas_sorted[-1] * 1.1],
        ]
    ),
    ncolors=len(Cas_sorted),
)

In [ ]:
for TARGET in y.columns:

    fig, axes = plt.subplots(
        1, len(slurries), sharex=True, sharey="row", figsize=(10, 5)
    )

    for slurry, ax in zip(slurries, axes):
        ok = data["slurry"] == slurry
        subdata = data[ok]

        for ca in subdata["capillary_number"].unique():
            ok = subdata["capillary_number"] == ca

            ax.scatter(
                subdata[ok]["gap_to_thickness_ratio"],
                subdata[ok][TARGET],
                color=cmap(norm(ca)),
            )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(
        sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
    )
    cbar.set_label("Ca")
    quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
    nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
    cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
    cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

    fig.supxlabel("Rgt")
    fig.supylabel(TARGET)
    fig.show()